# Tugas 1C: Advanced TF-IDF & Text Summarization (Padel Jakarta)

**Objective:** Menganalisis berita polemik lapangan padel Jakarta menggunakan perhitungan TF-IDF manual dan implementasi Text Summarization.

## 1. Persiapan Data & Library

In [ ]:
import pandas as pd
import math
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from wordcloud import WordCloud

news_content = """
JAKARTA: Polemik lapangan padel yang ramai diprotes warga membuat Pemerintah Provinsi (Pemprov) DKI Jakarta menetapkan lima aturan baru untuk menertibkan operasional dan perizinan fasilitas olahraga tersebut.
Keluhan warga di kawasan Cilandak dan Pulomas sebelumnya mencuat akibat kebisingan aktivitas padel yang berlangsung hingga malam hari. Lapangan padel yang berdiri di tengah permukiman dinilai mengganggu kenyamanan warga.
Pemprov DKI mencatat terdapat 397 lapangan padel yang tersebar di Jakarta. Saat ini pemerintah tengah menyisir kelengkapan perizinan, termasuk Persetujuan Bangunan Gedung (PBG).
Dari jumlah tersebut, sekitar setengahnya diduga belum memiliki PBG. Pendataan resmi masih menunggu hasil verifikasi Dinas Cipta Karya, Tata Ruang, dan Pertanahan.
Lapangan padel yang tidak memiliki PBG akan dikenai sanksi tegas, mulai dari penghentian kegiatan, pembongkaran bangunan, hingga pencabutan izin usaha.
Gubernur DKI Jakarta Pramono Anung menyampaikan keputusan tersebut usai memimpin rapat terbatas di Balai Kota Jakarta, Selasa (24/2).
Pertama, Pemprov menghentikan penerbitan izin pembangunan lapangan padel baru di zona perumahan. Ke depan, lapangan padel hanya diperbolehkan berdiri di kawasan komersial.
Kedua, lapangan padel yang sudah memiliki PBG tetapi berada di kawasan perumahan tetap diperbolehkan beroperasi, dengan pembatasan jam maksimal hingga pukul 20.00 WIB dan wajib dilengkapi peredam suara.
Ketiga, pembangunan lapangan padel baru wajib mendapatkan persetujuan teknis awal dari Dinas Pemuda dan Olahraga (Dispora) Jakarta.
Keempat, lapangan padel tidak diperkenankan berdiri di atas aset pemerintah yang berstatus Ruang Terbuka Hijau (RTH).
Kelima, Pemprov akan menertibkan persoalan parkir karena parkir sembarangan oleh pemain padel menjadi salah satu keluhan utama warga selain kebisingan dan jam operasional.
"""

sentences = [s.strip() for s in news_content.split('.') if len(s.strip()) > 15]
stopwords = set(["dan", "atau", "untuk", "di", "ke", "dari", "ini", "itu", "adalah", "akan", "dapat", "pada", "yang", "dengan"])
print(f"Berita dimuat dengan {len(sentences)} kalimat.")

## 2. Preprocessing & Regex Cleaning

In [ ]:
def preprocess(text, remove_stop=True):
    text = re.sub(r'[^\w\s]', '', text.lower())
    tokens = text.split()
    if remove_stop:
        tokens = [t for t in tokens if t not in stopwords]
    return tokens

tokenized_sentences = [preprocess(s) for s in sentences]
vocab = sorted(list(set([word for doc in tokenized_sentences for word in doc])))
print(f"Vocab size: {len(vocab)}")

## 3. Perhitungan Manual TF-IDF

In [ ]:
def get_tf_matrix(token_docs, vocabulary):
    tf_matrix = []
    for doc in token_docs:
        row = []
        total_terms = len(doc)
        for word in vocabulary:
            count = doc.count(word)
            row.append(count / total_terms if total_terms > 0 else 0)
        tf_matrix.append(row)
    return np.array(tf_matrix)

def get_idf_vector(token_docs, vocabulary):
    N = len(token_docs)
    idf_vector = []
    for word in vocabulary:
        df = sum(1 for doc in token_docs if word in doc)
        idf_vector.append(math.log10(N / df))
    return np.array(idf_vector)

tf_matrix = get_tf_matrix(tokenized_sentences, vocab)
idf_vector = get_idf_vector(tokenized_sentences, vocab)
tfidf_manual = tf_matrix * idf_vector

df_tfidf_manual = pd.DataFrame(tfidf_manual, columns=vocab)
display(df_tfidf_manual[vocab[:5]].head())

## 4. Analisis Kata Spesifik: 'padel'

In [ ]:
target_word = 'padel'
if target_word in df_tfidf_manual.columns:
    word_scores = df_tfidf_manual[target_word]
    print(f"Skor TF-IDF Manual untuk kata '{target_word}':")
    display(word_scores.head())
    
    plt.figure(figsize=(10, 4))
    sns.lineplot(x=word_scores.index, y=word_scores.values, marker='o', color='green')
    plt.title(f"TF-IDF Score of '{target_word}' per Sentence")
    plt.show()

## 5. Implementasi Text Summarization (Kalimat Terpenting)
Menampilkan 5 kalimat yang memiliki kepentingan (skor TF-IDF rata-rata) tertinggi.

In [ ]:
sentence_scores = []
for i in range(len(sentences)):
    scores = tfidf_manual[i]
    avg_score = np.mean(scores[scores > 0]) if any(scores > 0) else 0
    sentence_scores.append(avg_score)

df_summary = pd.DataFrame({'Original Sentence': sentences, 'Importance Score': sentence_scores}).sort_values(by='Importance Score', ascending=False)

print("TOP 5 RINGKASAN BERITA PADEL (SUMMARIZATION):")
display(df_summary.head(5))

## 6. Perbandingan dengan Library (TfidfVectorizer)

In [ ]:
vectorizer = TfidfVectorizer(stop_words=list(stopwords))
tfidf_lib = vectorizer.fit_transform(sentences)

df_tfidf_lib = pd.DataFrame(tfidf_lib.toarray(), columns=vectorizer.get_feature_names_out())
print("Hasil TF-IDF dari Scikit-Learn Library (Sample):")
display(df_tfidf_lib[sorted(list(set(df_tfidf_lib.columns) & set(vocab)))[:5]].head())

## 7. WordCloud Visualization

In [ ]:
all_text = " ".join([" ".join(s) for s in tokenized_sentences])
wc = WordCloud(width=800, height=400, background_color='white', colormap='viridis').generate(all_text)
plt.figure(figsize=(12, 6))
plt.imshow(wc)
plt.axis('off')
plt.show()